In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = r'C:\FRAUD-Sinta2\Data\Olist'
OUTPUT_PATH = r'C:\FRAUD-Sinta2\Output'

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Libraries imported and setup complete.")

Libraries imported and setup complete.


In [2]:
try:
    df_orders = pd.read_csv(os.path.join(DATA_PATH, 'olist_orders_dataset.csv'))
    df_payments = pd.read_csv(os.path.join(DATA_PATH, 'olist_order_payments_dataset.csv'))
    print("Dataset Olist berhasil di-load.")
except FileNotFoundError:
    print(f"Error: Pastikan dataset Olist ada di folder: {DATA_PATH}")

# Gabungkan & Ganti Nama
df_orders = df_orders[['order_id', 'customer_id', 'order_purchase_timestamp']]
df_payments = df_payments[['order_id', 'payment_value']]
df_merged = pd.merge(df_orders, df_payments, on='order_id', how='left')
df_merged.rename(columns={
    'customer_id': 'user_id',
    'order_purchase_timestamp': 'timestamp',
    'payment_value': 'amount'
}, inplace=True)

# Bersihkan
df_merged['timestamp'] = pd.to_datetime(df_merged['timestamp'])
df_merged['amount'].fillna(0, inplace=True)
df_merged.dropna(inplace=True)
df_merged.sort_values(by=['user_id', 'timestamp'], inplace=True)
df_merged.drop_duplicates(inplace=True)
df_merged.reset_index(drop=True, inplace=True)

print(f"Data Olist digabungkan & dibersihkan. Shape: {df_merged.shape}")


Dataset Olist berhasil di-load.


C:\Users\yosua\AppData\Local\Temp\ipykernel_5100\2804950866.py:20: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_merged['amount'].fillna(0, inplace=True)


Data Olist digabungkan & dibersihkan. Shape: (103272, 4)


In [3]:
print("Memulai User-Based Split untuk mencegah data bocor...")
# 1. Dapatkan daftar user unik
all_users = df_merged['user_id'].unique()
n_users = len(all_users)
print(f"Total user unik: {n_users}")

# 2. Acak daftar user
np.random.shuffle(all_users)

# 3. Tentukan titik split (70% Train, 15% Val, 15% Test)
train_split_idx = int(n_users * 0.70)
val_split_idx = int(n_users * 0.85)

train_users = all_users[:train_split_idx]
val_users = all_users[train_split_idx:val_split_idx]
test_users = all_users[val_split_idx:]

print(f"User di Train Set: {len(train_users)}")
print(f"User di Val Set: {len(val_users)}")
print(f"User di Test Set: {len(test_users)}")

# 4. Buat 3 DataFrame terpisah
train_df = df_merged[df_merged['user_id'].isin(train_users)].copy()
val_df = df_merged[df_merged['user_id'].isin(val_users)].copy()
test_df = df_merged[df_merged['user_id'].isin(test_users)].copy()

# 5. Verifikasi (PENTING)
# Cek apakah ada user yang 'bocor' antar set
train_users_set = set(train_df['user_id'])
val_users_set = set(val_df['user_id'])
test_users_set = set(test_df['user_id'])

print(f"Kebocoran Train-Val: {len(train_users_set.intersection(val_users_set))}")
print(f"Kebocoran Val-Test: {len(val_users_set.intersection(test_users_set))}")
print(f"Kebocoran Train-Test: {len(train_users_set.intersection(test_users_set))}")
# (Hasilnya harus 0 0 0)

print("\nSplit berbasis user selesai.")

Memulai User-Based Split untuk mencegah data bocor...
Total user unik: 99441
User di Train Set: 69608
User di Val Set: 14916
User di Test Set: 14917
Kebocoran Train-Val: 0
Kebocoran Val-Test: 0
Kebocoran Train-Test: 0

Split berbasis user selesai.


In [4]:
def inject_anomalies_v3_1(df_base, n_anomalies_per_type=150):
    
    # 1. Ambil semua data normal (transaksi > 0)
    # Ini adalah 'kolam' data normal kita untuk disuntik anomali
    df_normal_pool = df_base[df_base['amount'] > 0].copy()
    df_normal_pool['Class'] = 0
    df_normal_pool['anomaly_type'] = 0
    
    print(f"  > Ukuran 'kolam' data normal: {len(df_normal_pool)}")
    if df_normal_pool.empty:
        print("  > Peringatan: Tidak ada data normal di set ini. Mengembalikan kosong.")
        return df_normal_pool
        
    all_anomalies = [] # Tempat menampung anomali
    
    # Ambil statistik amount per user DARI SEMUA USER DI SET INI
    user_stats = df_normal_pool.groupby('user_id')['amount'].agg(['mean', 'std']).fillna(0)
    
    # ----------------------------------------------------
    # --- Tipe 1: Anomali Nilai (Value Anomaly) - Halus ---
    # ----------------------------------------------------
    
    # (PERBAIKAN LOGIKA)
    # Pilih user acak dari user_stats (yang sudah pasti punya trx > 0)
    n_users_available = len(user_stats)
    n_anom_1 = min(n_anomalies_per_type, n_users_available)
    if n_anom_1 == 0:
        print("  > Tipe 1 (Nilai Halus) dilewati: Tidak ada user yang tersedia.")
    else:
        # replace=False agar user unik
        users_to_make_anom_1 = np.random.choice(
            user_stats.index, n_anom_1, replace=False 
        )
        
        anom_1_list = []
        for user_id in users_to_make_anom_1:
            # Ambil 1 transaksi acak dari user ini
            anom_trx = df_normal_pool[df_normal_pool['user_id'] == user_id].sample(1).copy()
            
            # Dapatkan statistik user tersebut
            stats = user_stats.loc[user_id]
            user_mean = stats['mean']
            user_std = stats['std']
            
            # Hitung anomali (4x std dev, atau 1000 jika std=0)
            anom_amount = user_mean + (4 * user_std) + np.random.uniform(1, 100)
            if user_std == 0: # Jika user hanya punya 1 trx, std-nya 0
                anom_amount = user_mean * 10 + np.random.uniform(1, 100) # Buat anomali yg wajar

            anom_trx['amount'] = anom_amount
            anom_trx['Class'] = 1
            anom_trx['anomaly_type'] = 1 # Tipe 1 = Nilai
            anom_1_list.append(anom_trx)
            
        if anom_1_list:
            df_anom_1 = pd.concat(anom_1_list)
            all_anomalies.append(df_anom_1)
            print(f"  > Tipe 1 (Nilai Halus) dibuat: {len(df_anom_1)} sampel")
    
    # --------------------------------------------------------
    # --- Tipe 3: Anomali Sekuensial (Sequential) - Halus ---
    # --------------------------------------------------------
    n_anom_3 = min(n_anomalies_per_type, n_users_available)
    if n_anom_3 == 0:
        print("  > Tipe 3 (Sekuensial Halus) dilewati: Tidak ada user yang tersedia.")
    else:
        users_to_make_anom_3 = np.random.choice(
            user_stats.index, n_anom_3, replace=False
        )
        
        anom_3_list = []
        for user_id in users_to_make_anom_3:
            template_trx = df_normal_pool[df_normal_pool['user_id'] == user_id].sample(1).copy()
            
            start_time = pd.to_datetime(template_trx['timestamp'].iloc[0])
            start_amount = np.random.uniform(500, 2000) # Mulai dari amount yg besar
            
            for i in range(5): # Buat 5 transaksi
                new_trx = template_trx.copy()
                
                # --- PERBAIKAN BUG 'VALUEERROR' DI SINI ---
                # 1. Hitung 'new_amount' sebagai ANGKA BIASA (scalar)
                new_amount_value = start_amount - (i * np.random.uniform(50, 100))
                
                # 2. Lakukan 'if' pada ANGKA BIASA
                if new_amount_value < 0:
                    new_amount_value = 10.0 # Jaga agar tidak negatif
                
                # 3. SETELAH aman, masukkan ke DataFrame
                new_trx['amount'] = new_amount_value
                # --- PERBAIKAN SELESAI ---
                
                # Terjadi dalam 1 jam (tambah 0-3600 detik)
                new_trx['timestamp'] = start_time + pd.to_timedelta(np.random.randint(0, 3600), unit='s')
                new_trx['Class'] = 1
                new_trx['anomaly_type'] = 3 # Tipe 3 = Sekuensial
                anom_3_list.append(new_trx)
                
        if anom_3_list:
            df_anom_3 = pd.concat(anom_3_list)
            all_anomalies.append(df_anom_3)
            print(f"  > Tipe 3 (Sekuensial Halus) dibuat: {len(df_anom_3)} sampel")

    # Gabungkan semua
    df_final = pd.concat([df_normal_pool] + all_anomalies)
    # Acak data
    df_final = df_final.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    return df_final

In [5]:
# 1. Buat Train Set (100% Normal)
# Kita ambil HANYA transaksi normal dari user train
df_train_final = train_df[train_df['amount'] > 0].copy()
df_train_final['Class'] = 0
df_train_final['anomaly_type'] = 0
print(f"Train Set (Normal) Dibuat. Shape: {df_train_final.shape}")

# 2. Buat Validation Set (Normal + Anomali Halus)
print("\nMembuat Validation Set...")
df_val_final = inject_anomalies_v3_1(val_df, n_anomalies_per_type=50) # <-- Panggil fungsi baru
print(f"Validation Set (Campuran) Dibuat. Shape: {df_val_final.shape}")

# 3. Buat Test Set (Normal + Anomali Halus)
print("\nMembuat Test Set...")
df_test_final = inject_anomalies_v3_1(test_df, n_anomalies_per_type=50) # <-- Panggil fungsi baru
print(f"Test Set (Campuran) Dibuat. Shape: {df_test_final.shape}")

# 4. Simpan ke CSV
df_train_final.to_csv(os.path.join(OUTPUT_PATH, 'train_v3.csv'), index=False)
df_val_final.to_csv(os.path.join(OUTPUT_PATH, 'val_v3.csv'), index=False)
df_test_final.to_csv(os.path.join(OUTPUT_PATH, 'test_v3.csv'), index=False)

print("\nSEMUA DATASET V3 BARU TELAH DISIMPAN.")

# (Tambahkan pengecekan jika test_df_final tidak kosong)
if not df_test_final.empty:
    print("\nDistribusi Kelas Test Set:")
    print(df_test_final['Class'].value_counts(normalize=True))
    print("\nDistribusi Tipe Anomali Test Set:")
    print(df_test_final['anomaly_type'].value_counts(normalize=True))
else:
    print("\nPERINGATAN: Test Set kosong, tidak ada anomali yang dibuat.")


Train Set (Normal) Dibuat. Shape: (72323, 6)

Membuat Validation Set...
  > Ukuran 'kolam' data normal: 15450
  > Tipe 1 (Nilai Halus) dibuat: 50 sampel
  > Tipe 3 (Sekuensial Halus) dibuat: 250 sampel
Validation Set (Campuran) Dibuat. Shape: (15750, 6)

Membuat Test Set...
  > Ukuran 'kolam' data normal: 15490
  > Tipe 1 (Nilai Halus) dibuat: 50 sampel
  > Tipe 3 (Sekuensial Halus) dibuat: 250 sampel
Test Set (Campuran) Dibuat. Shape: (15790, 6)

SEMUA DATASET V3 BARU TELAH DISIMPAN.

Distribusi Kelas Test Set:
Class
0    0.981001
1    0.018999
Name: proportion, dtype: float64

Distribusi Tipe Anomali Test Set:
anomaly_type
0    0.981001
3    0.015833
1    0.003167
Name: proportion, dtype: float64
